In [2]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
!pip install torchmetrics transformers
import torchmetrics

import urllib.request

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 21.5 MB/s eta 0:00:0000:01


### Load the dataset

In [3]:
url = "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"
csv_path = 'IMDB-Dataset.csv'
if not os.path.exists(csv_path):
  !wget -O IMDB-Dataset.csv -q "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"
  # urllib.request.urlretrieve(url, csv_path)

In [4]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


####  Preprocess the dataset

In [5]:
# remove line break tags
text = list(df['review'].str.replace('<br />',''))

# remap labels to 0 (negative) and 1 (positive)
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

#### Make train/test split

In [6]:
from sklearn.model_selection import train_test_split
# 90/10 train/test split
text_train, text_test, labels_train, labels_test = train_test_split(text,labels,test_size=0.1,random_state=42)

### Bag-of-words experiment

Here you can implement the bag-of-words experiment using scikit-learn.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
vectorizer.fit(text_train)
X_train = vectorizer.transform(text_train)
X_test = vectorizer.transform(text_test)
y_train = labels_train
y_test = labels_test
vectorizer.get_feature_names_out()

print(f'X_train shape: {X_train.shape} of type {type(X_train)}')
print(f'X_test shape: {X_test.shape} of type {type(X_test)}')
print(f'y_train shape: {y_train.shape} of type {type(y_train)}')
print(f'y_test shape: {y_test.shape} of type {type(y_test)}')

X_train shape: (45000, 1000) of type <class 'scipy.sparse._csr.csr_matrix'>
X_test shape: (5000, 1000) of type <class 'scipy.sparse._csr.csr_matrix'>
y_train shape: (45000,) of type <class 'numpy.ndarray'>
y_test shape: (5000,) of type <class 'numpy.ndarray'>


In [8]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(verbose=True).fit(X_train, y_train)

Iteration 1, loss = 0.45291776
Iteration 2, loss = 0.31147217
Iteration 3, loss = 0.29908185
Iteration 4, loss = 0.29557072
Iteration 5, loss = 0.29325389
Iteration 6, loss = 0.29066121
Iteration 7, loss = 0.28778491
Iteration 8, loss = 0.28491225
Iteration 9, loss = 0.28151550
Iteration 10, loss = 0.27628456
Iteration 11, loss = 0.27240924
Iteration 12, loss = 0.26762822
Iteration 13, loss = 0.26295352
Iteration 14, loss = 0.25778370
Iteration 15, loss = 0.25307182
Iteration 16, loss = 0.24781922
Iteration 17, loss = 0.24241015
Iteration 18, loss = 0.23691400
Iteration 19, loss = 0.23154151
Iteration 20, loss = 0.22579796
Iteration 21, loss = 0.21897028
Iteration 22, loss = 0.21235213
Iteration 23, loss = 0.20651771
Iteration 24, loss = 0.19850456
Iteration 25, loss = 0.19139655
Iteration 26, loss = 0.18346702
Iteration 27, loss = 0.17539684
Iteration 28, loss = 0.16731517
Iteration 29, loss = 0.15868244
Iteration 30, loss = 0.15043683
Iteration 31, loss = 0.14185245
Iteration 32, los

In [9]:
print(f'The final MLP train accuracy is: {model.score(X_train, y_train)}')
print(f'The final MLP test accuracy is:  {model.score(X_test, y_test)}')

The final MLP train accuracy is: 1.0
The final MLP test accuracy is:  0.862


### RNN Experiment

#### Tokenize the texts

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# get number of tokens in vocabulary
vocab_size = len(tokenizer.vocab)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [11]:
tokenized_text_train = [tokenizer(t)['input_ids'] for t in text_train]
tokenized_text_test = [tokenizer(t)['input_ids'] for t in text_test]

Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


#### Custom Dataset class

This is a custom subclass of Dataset that will produce sequences of tokens and associated sentiment labels.

If a maximum sequence length is provided, it will randomly truncate the text or zero pad it as necessary.

In [12]:
class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]

    # randomly truncate or zero pad if necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        # choose random substring
        ind = np.random.randint(len(token_ids)-self.max_seq_len)
        token_ids = token_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
    
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(self.labels[idx])


In [13]:
train_ds = TokenDataset(tokenized_text_train,labels_train,max_seq_len=100)
test_ds = TokenDataset(tokenized_text_test,labels_test)

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
test_loader = DataLoader(test_ds,batch_size=1,shuffle=False)

Here you can implement the RNN experiment.

In [14]:
x_batch, y_batch = next(iter(train_loader))
print(f'x_batch has shape {x_batch.shape} and values:\n\t{x_batch}')
print(f'y_batch has shape {y_batch.shape} and values:\n\t{y_batch}')

x_batch has shape torch.Size([32, 100]) and values:
	tensor([[20866,  1107, 11918,  ...,   170,  5263,   117],
        [ 1110,   131,   170,  ...,  1134,  1119,  1169],
        [  119,   119,  3774,  ...,  1250,   117,  1185],
        ...,
        [ 8005,  3962,  4204,  ...,  1193,  1176,   170],
        [23808,  4069,  1106,  ...,  1261,  1143,  1118],
        [ 1622,  1175,  1122,  ...,  8305,  1959,   113]])
y_batch has shape torch.Size([32]) and values:
	tensor([0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 1, 1])


In [15]:
device = "cuda"
# device = "cpu"

In [16]:
class GRUSequenceBinary(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)  # 2 logits
          
    def forward(self, x):
        x = self.embed(x)
        out, h_n = self.gru(x)
        last = h_n[-1]            # [B, H]
        logits = self.fc(last)    # [B, 2]
        return logits

In [22]:
import torch.nn as nn
from torchmetrics.classification import BinaryAccuracy

embed_size = 101
hidden_size = 102
model = GRUSequenceBinary(vocab_size, embed_size, hidden_size, num_layers=3)
model = model.to(device)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-6)
metric = BinaryAccuracy()
metric = metric.to(device)

In [ ]:
def compute_model_acc(model: nn.Sequential | nn.Module, loader: DataLoader, metric):
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            z_batch = model(X_batch)
            y_predict = torch.argmax(z_batch, dim=1)
            metric.update(y_predict, y_batch)
    overall_acc = metric.compute()
    metric.reset()
    return overall_acc

In [25]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        opt.zero_grad()
        z_batch = model(X_batch)
        loss = loss_fn(z_batch, y_batch)
        loss.backward()
        opt.step()

        # calculate accuracy
        y_predict = torch.argmax(z_batch, dim=1)
        metric.update(y_predict, y_batch)

    epoch_train_acc = metric.compute()
    metric.reset()
    print(f'epoch {epoch}: loss is {loss.item():.4f} -- training accuracy is {epoch_train_acc:.4f}, test accuracy is {compute_model_acc(model, test_loader, metric):.4f}')

epoch 0: loss is 0.2273 -- training accuracy is 0.7711, test accuracy is 0.8590
epoch 1: loss is 0.6245 -- training accuracy is 0.8097, test accuracy is 0.8560
epoch 2: loss is 0.2315 -- training accuracy is 0.8285, test accuracy is 0.8862
epoch 3: loss is 0.4017 -- training accuracy is 0.8409, test accuracy is 0.8886
epoch 4: loss is 0.1519 -- training accuracy is 0.8532, test accuracy is 0.9012
epoch 5: loss is 0.6570 -- training accuracy is 0.8628, test accuracy is 0.8930
epoch 6: loss is 0.0509 -- training accuracy is 0.8701, test accuracy is 0.8946
epoch 7: loss is 0.4009 -- training accuracy is 0.8787, test accuracy is 0.9012
epoch 8: loss is 0.5935 -- training accuracy is 0.8837, test accuracy is 0.9018
epoch 9: loss is 0.1044 -- training accuracy is 0.8910, test accuracy is 0.9090


In [26]:
print(f'The final GRU train accuracy is: {compute_model_acc(model, train_loader, metric)}')
print(f'The final GRU test accuracy is:  {compute_model_acc(model, test_loader, metric)}')

The final GRU train accuracy is: 0.8992666602134705
The final GRU test accuracy is:  0.9089999794960022
